## Section 1: Importing the Libraries

In [ ]:
import time
import torch
#used to create the generator and discriminator
from torch import nn 
# used to create visualisations
import matplotlib.pyplot as plt 
# used to set size of visualisations
from matplotlib import rcParams 
#used to clear the visualisations at the end of each epoch
from IPython import display 
#saving outputs to the notebook
%matplotlib inline

## Creating the Data

The generated dataset approximates a linear relationship between two variables X and Y using the formula
$$Y = XA + b$$
which is simply a vectorized version of linear regression

#### Creating the dataset

In [ ]:
X = torch.normal(0.0, 1, (1000, 2))
A = torch.tensor([[1, 2], [-0.1, 0.5]])
b = torch.tensor([1, 2])
data = torch.matmul(X, A) + b

#### Display dataset

In [ ]:
plt.scatter(data[:100, 0].detach().numpy(), data[:100, 1].detach().numpy())

#### Creating Dataset Iterateables

In [ ]:
batch_size = 8
dataset = torch.utils.data.TensorDataset(data)
data_iter = torch.utils.data.DataLoader(dataset, batch_size, shuffle=True)


## Section 2: The Model
### Generators and Discriminators
The generator needs to be trained to generator the discriminator cannot classify as fake. Sine the dataset is created using linear rigression, we use a single layered neural network.

![alt text](images/generator.png "Generator")

The discriminator is a slightly more complex neural network since it has to learn to classify the generated examples as real or fake.

![alt text](images/discriminator.png "Discriminator")

#### Generator Neural Net

In [ ]:
nnet_Gen = nn.Sequential(nn.Linear(2, 2))

#### Discriminator Neural Net

In [ ]:
nnet_Disc = nn.Sequential(
  nn.Linear(2, 5), nn.Tanh(),
  nn.Linear(5, 3), nn.Tanh(),
  nn.Linear(3, 1))

### Discriminator Updates

we use the real and synthetic batches to compute the loss by computing the loss on the discriminator outputs for real batch with a tensor of ones which denote real samples, and for synthetic batch with a tensor of zeroes which denote fakes. The final loss is an average of the two, since the discriminator's job is to correctly classify the two distributions as separate.

In [ ]:
"""
X is the training batch
Z is the seed values for the fake batch
nnet_D is the discriminator neural net
nnet_G is the generator neural net
loss is the loss function
trainer_d is the optimizer
"""
def update_D(X, Z, nnet_D, nnet_G, loss, trainer_D):
    batch_size = X.shape[0]
    ones = torch.ones((batch_size,), device=X.device)
    zeros = torch.zeros((batch_size,), device=X.device)
    trainer_D.zero_grad()
    real_Y = nnet_D(X)
    synth_X = nnet_G(Z)
    synth_Y = nnet_D(synth_X.detach())
    loss_D = (loss(real_Y, ones.reshape(real_Y.shape)) +
              loss(synth_Y, zeros.reshape(synth_Y.shape))) / 2
    loss_D.backward()
    trainer_D.step()
    return loss_D


### Generator Updates

We use the noise given as input to generate a synthetic batch and make predictions using the discriminator then compute the loss on these predictions with ones (to fool the discriminator better), compute the gradients, and perform updates. 

In [ ]:
"""
Z is the noise vector for generating fake branch
nnet_D is the discriminator neural net
nnet_G is the generator neural net
loss is the loss function
trainer_G is the optimizer
"""
def update_G(Z, nnet_D, nnet_G, loss, trainer_G):
    batch_size = Z.shape[0]
    ones = torch.ones((batch_size,), device=Z.device)
    trainer_G.zero_grad()
    synth_X = nnet_G(Z)
    synth_Y = nnet_D(synth_X)
    loss_G = loss(synth_Y, ones.reshape(synth_Y.shape))
    loss_G.backward()
    trainer_G.step()
    return loss_G


## Section 3: The Train Function
### Initialize the Parameters

In [ ]:
"""
Discriminator is the discriminator neural net
Generator is the generator neural net
lr_D is the learning rate for the discriminator
lr_G is the learning rate for the generator
"""
def init_params(Discriminator, Generator, lr_D, lr_G):
    loss = nn.BCEWithLogitsLoss(reduction='sum')
    for w in Discriminator.parameters():
        nn.init.normal_(w, 0, 0.02)
    for w in Generator.parameters():
        nn.init.normal_(w, 0, 0.02)
    trainer_D = torch.optim.Adam(Discriminator.parameters(), lr=lr_D)
    trainer_G = torch.optim.Adam(Generator.parameters(), lr=lr_G)
    fig, axes = plt.subplots(2, 1, figsize=(5, 9))
    loss_D = []
    loss_G = []

    return loss, trainer_D, trainer_G, fig, axes, loss_D, loss_G

"""
loss is the loss function
trainer_D is the optimizer for the discriminator
trainer_G is the optimizer for the generator
fig is the figure for visualisations
axes is the axes for plotting the visualisations
loss_D is the list of discriminator losses
loss_G is the list of generator losses
"""